# Task 4 – Data Profiling & Validation

## Overview
This notebook validates data quality across 4 RecoMart data sources:
1. **User Interaction Logs** (clickstream data)
2. **Purchase History** (transaction data)
3. **Product Catalog** (item metadata)
4. **External Signals** (popularity metrics)

## Root Causes Fixed
1. **Session timeout** → All checks per table batched into ONE `agg()` call
2. **Pandas on Spark** → Removed all pandas-style calls, replaced with pure PySpark
3. **len(df) on Spark** → Replaced with `df.count()` called once and reused
4. **Wrong schema ref** → Fixed column references
5. **Empty string check** → Only checks empty string for StringType columns, preventing cast errors

## Strategy
For every table, run ONE agg() pass that computes:
- Missing counts for all columns
- Domain violation counts
- Range violation counts
- Distinct counts (cardinality)
- Numeric stats (min/max/mean/median/std)
- Duplicate counts

This reduces ~15 Spark jobs per table → 2-3 jobs per table.

## Output
Validation results are saved to: `workspace.recomart_silver.validation_data_json`

In [0]:
import json
from datetime import datetime
import pyspark.sql.functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType
from typing import List, Dict, Any

# Initialize results dictionary
results: Dict[str, Any] = {}

In [0]:
def pct(n: int, total: int) -> float:
    """Calculate percentage with 2 decimal places."""
    return round(100 * n / total, 2) if total else 0.0

In [0]:
def run_all_checks(df, cols, numeric_cols, cardinality_cols, domain_checks, range_checks_def, pk_col):
    """
    Single-pass aggregation over df that computes EVERYTHING in 2 Spark jobs:
      Job 1 → df.count()
      Job 2 → df.agg(all expressions at once)
    
    Then cleans the dataframe by:
      1. Removing rows with missing values in critical columns
      2. Removing domain violations
      3. Removing range violations
      4. Removing duplicates
    
    Args:
        df: Input DataFrame
        cols: List of all column names
        numeric_cols: List of numeric columns to compute stats for
        cardinality_cols: List of columns to compute distinct counts for
        domain_checks: List of dicts with 'name' and 'condition' keys
        range_checks_def: List of dicts with 'col', 'lo', 'hi' keys
        pk_col: Primary key column name
    
    Returns:
        tuple: (validation_results_dict, cleaned_dataframe)
    """
    total = df.count()                       # Job 1 — only count ever called alone
    df_original = df  # Keep reference to original

    # ── Build ALL aggregation expressions in one pass ───────────────────────────
    agg_exprs = []
    
    # Get schema for type checking
    field_types = {f.name: f.dataType for f in df.schema.fields}

    # 1. Missing counts per column - only check empty string for string columns
    for c in cols:
        if isinstance(field_types.get(c), StringType):
            # For string columns, check both null and empty string
            agg_exprs.append(
                F.sum(F.when(F.col(c).isNull() | (F.col(c) == ""), 1).otherwise(0))
                 .alias(f"__miss_{c}")
            )
        else:
            # For non-string columns, only check null
            agg_exprs.append(
                F.sum(F.when(F.col(c).isNull(), 1).otherwise(0))
                 .alias(f"__miss_{c}")
            )

    # 2. Numeric stats
    for c in numeric_cols:
        agg_exprs += [
            F.min(F.col(c).cast("double")).alias(f"__min_{c}"),
            F.max(F.col(c).cast("double")).alias(f"__max_{c}"),
            F.mean(F.col(c).cast("double")).alias(f"__mean_{c}"),
            F.percentile_approx(F.col(c).cast("double"), 0.5).alias(f"__med_{c}"),
            F.stddev(F.col(c).cast("double")).alias(f"__std_{c}"),
        ]

    # 3. Cardinality (distinct counts)
    for c in cardinality_cols:
        agg_exprs.append(
            F.countDistinct(F.col(c)).alias(f"__card_{c}")
        )

    # 4. Domain violation counts  (each is a conditional sum)
    for chk in domain_checks:
        agg_exprs.append(
            F.sum(F.when(chk["condition"], 1).otherwise(0))
             .alias(f"__dom_{chk['name']}")
        )

    # 5. Range violation counts
    for rchk in range_checks_def:
        c, lo, hi = rchk["col"], rchk["lo"], rchk["hi"]
        agg_exprs.append(
            F.sum(
                F.when(
                    F.col(c).isNotNull() &
                    ((F.col(c).cast("double") < lo) | (F.col(c).cast("double") > hi)),
                    1
                ).otherwise(0)
            ).alias(f"__range_{c}")
        )

    # 6. Full-row duplicate count  (total - distinct)
    agg_exprs.append(F.countDistinct(*[F.col(c) for c in cols]).alias("__full_distinct"))

    # 7. PK duplicate count
    if pk_col:
        agg_exprs.append(F.countDistinct(F.col(pk_col)).alias("__pk_distinct"))

    # ── Single agg() call — Job 2 ──────────────────────────────────────────────
    row = df.agg(*agg_exprs).collect()[0].asDict()

    # ── Unpack results ──────────────────────────────────────────────────────────
    missing = []
    for c in cols:
        n = int(row.get(f"__miss_{c}") or 0)
        missing.append({"column": c, "missing_count": n, "missing_pct": pct(n, total)})

    num_stats = []
    for c in numeric_cols:
        num_stats.append({
            "column": c,
            "min":    row.get(f"__min_{c}"),
            "max":    row.get(f"__max_{c}"),
            "mean":   round(float(row.get(f"__mean_{c}") or 0), 4),
            "median": row.get(f"__med_{c}"),
            "std":    round(float(row.get(f"__std_{c}") or 0), 4),
        })

    cardinality = {c: int(row.get(f"__card_{c}") or 0) for c in cardinality_cols}

    domain_results = []
    for chk in domain_checks:
        v = int(row.get(f"__dom_{chk['name']}") or 0)
        domain_results.append({
            "check":      chk["name"],
            "violations": v,
            "status":     "PASS" if v == 0 else "FAIL",
        })

    range_results = []
    for rchk in range_checks_def:
        v = int(row.get(f"__range_{rchk['col']}") or 0)
        range_results.append({
            "col":        rchk["col"],
            "expected":   f"[{rchk['lo']}, {rchk['hi']}]",
            "violations": v,
            "status":     "PASS" if v == 0 else "FAIL",
        })

    full_distinct = int(row.get("__full_distinct") or 0)
    full_dupes    = max(0, total - full_distinct)

    pk_distinct = int(row.get("__pk_distinct") or 0) if pk_col else total
    pk_dupes    = max(0, total - pk_distinct)

    # ── DATA CLEANING ───────────────────────────────────────────────────────────
    df_clean = df_original
    rows_removed = {"missing": 0, "domain": 0, "range": 0, "duplicates": 0}
    
    # Step 1: Remove rows with missing values in key columns (non-metadata columns)
    # Skip metadata columns that start with underscore
    key_cols = [c for c in cols if not c.startswith("_")]
    for c in key_cols:
        before = df_clean.count()
        if isinstance(field_types.get(c), StringType):
            df_clean = df_clean.filter(
                F.col(c).isNotNull() & (F.col(c) != "")
            )
        else:
            df_clean = df_clean.filter(F.col(c).isNotNull())
        rows_removed["missing"] += (before - df_clean.count())
    
    # Step 2: Remove domain violations
    for chk in domain_checks:
        before = df_clean.count()
        # Keep rows that DO NOT violate (negate the violation condition)
        df_clean = df_clean.filter(~chk["condition"])
        rows_removed["domain"] += (before - df_clean.count())
    
    # Step 3: Remove range violations
    for rchk in range_checks_def:
        c, lo, hi = rchk["col"], rchk["lo"], rchk["hi"]
        before = df_clean.count()
        df_clean = df_clean.filter(
            F.col(c).isNull() | 
            ((F.col(c).cast("double") >= lo) & (F.col(c).cast("double") <= hi))
        )
        rows_removed["range"] += (before - df_clean.count())
    
    # Step 4: Remove duplicate rows (keep first occurrence)
    if pk_col:
        # Remove duplicates based on primary key
        before = df_clean.count()
        df_clean = df_clean.dropDuplicates([pk_col])
        rows_removed["duplicates"] += (before - df_clean.count())
    else:
        # Remove full row duplicates
        before = df_clean.count()
        df_clean = df_clean.dropDuplicates()
        rows_removed["duplicates"] += (before - df_clean.count())
    
    final_count = df_clean.count()
    total_removed = sum(rows_removed.values())

    validation_results = {
        "total_rows":     total,
        "total_cols":     len(cols),
        "columns":        cols,
        "missing":        missing,
        "numeric_stats":  num_stats,
        "cardinality":    cardinality,
        "domain_checks":  domain_results,
        "range_checks":   range_results,
        "duplicates":     {"duplicate_rows": full_dupes, "status": "PASS" if full_dupes == 0 else "WARN"},
        "key_dup":        {"duplicate_rows": pk_dupes,   "status": "PASS" if pk_dupes == 0 else "WARN"},
        "cleaning_summary": {
            "original_rows": total,
            "final_rows": final_count,
            "total_removed": total_removed,
            "removed_by_reason": rows_removed,
            "retention_rate_pct": pct(final_count, total)
        }
    }
    
    return validation_results, df_clean

In [0]:
def get_distribution(df: DataFrame, col: str) -> Dict[str, int]:
    """Lightweight groupBy for value distributions — single Spark job."""
    return {
        row[col]: row["count"]
        for row in df.groupBy(col).count().collect()
        if row[col] is not None
    }


def dtype_map(cols: List[str], schema) -> Dict[str, str]:
    """Read dtypes from already-captured schema — zero RPCs."""
    field_map = {f.name: str(f.dataType) for f in schema.fields}
    return {c: field_map.get(c, "unknown") for c in cols}

In [0]:
print("Profiling: user_interaction_logs ...")

df_clickstream  = spark.table("workspace.recomart_bronze.user_interaction_logs")
click_cols      = df_clickstream.columns       # schema resolved once
click_schema    = df_clickstream.schema

VALID_EVENT_TYPES = ["view", "click", "add_to_cart", "purchase"]
VALID_DEVICES     = ["mobile", "desktop", "tablet"]

click_checks, df_clickstream_clean = run_all_checks(
    df          = df_clickstream,
    cols        = click_cols,
    numeric_cols= [],                          # no numeric range cols in clickstream
    cardinality_cols = ["user_id", "item_id", "event_type", "device", "city"],
    domain_checks = [
        {
            "name":      "event_type values",
            "condition": ~F.col("event_type").isin(VALID_EVENT_TYPES),
        },
        {
            "name":      "device values",
            "condition": ~F.col("device").isin(VALID_DEVICES),
        },
        {
            "name":      "user_id format",
            "condition": ~F.col("user_id").rlike(r"^USER_\d{4}$"),
        },
        {
            "name":      "item_id format",
            "condition": ~F.col("item_id").rlike(r"^ITEM_\d{4}$"),
        },
        {
            "name":      "event_id format",
            "condition": ~F.col("event_id").rlike(r"^EVT_\d{6}$"),
        },
        {
            "name":      "timestamp format",
            "condition": ~F.col("timestamp").rlike(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"),
        },
    ],
    range_checks_def = [],
    pk_col      = "event_id",
)

# Value distributions — one groupBy each, separated to keep agg readable
event_dist  = get_distribution(df_clickstream, "event_type")
device_dist = get_distribution(df_clickstream, "device")

results["clickstream"] = {
    "source": "user_interaction_logs",
    **click_checks,
    "dtypes":             dtype_map(click_cols, click_schema),
    "event_distribution": event_dist,
    "device_distribution":device_dist,
}

print(f"  rows={results['clickstream']['total_rows']:,}  "
      f"domain_checks={len(results['clickstream']['domain_checks'])}")
print(f"  Cleaned: {click_checks['cleaning_summary']['final_rows']:,} rows retained "
      f"({click_checks['cleaning_summary']['retention_rate_pct']}%)")

# Write cleaned data to silver table
df_clickstream_clean.write.mode("overwrite").saveAsTable("workspace.recomart_silver.user_interaction_logs_cleaned")
print("  ✅ Cleaned data saved to: workspace.recomart_silver.user_interaction_logs_cleaned")

In [0]:
print("Profiling: purchase_history ...")

df_transactions = spark.table("workspace.recomart_bronze.purchase_history")
purchase_cols   = df_transactions.columns
purchase_schema = df_transactions.schema

VALID_PAYMENTS = ["credit_card", "debit_card", "UPI", "COD", "wallet"]
VALID_GENDERS  = ["M", "F", "Other"]
VALID_AGES     = ["18-24", "25-34", "35-44", "45-54", "55+"]

txn_checks, df_transactions_clean = run_all_checks(
    df           = df_transactions,
    cols         = purchase_cols,
    numeric_cols = ["quantity", "unit_price", "total_price"],
    cardinality_cols = ["user_id", "item_id", "payment_method", "gender", "age_group"],
    domain_checks = [
        {
            "name":      "payment_method values",
            "condition": F.col("payment_method").isNotNull() &
                         ~F.col("payment_method").isin(VALID_PAYMENTS),
        },
        {
            "name":      "gender values",
            "condition": ~F.col("gender").isin(VALID_GENDERS),
        },
        {
            "name":      "age_group values",
            "condition": ~F.col("age_group").isin(VALID_AGES),
        },
    ],
    range_checks_def = [
        {"col": "quantity",    "lo": 1,    "hi": 100},
        {"col": "unit_price",  "lo": 0.01, "hi": 100000},
        {"col": "total_price", "lo": 0.01, "hi": 500000},
    ],
    pk_col = "order_id",
)

payment_dist = get_distribution(df_transactions, "payment_method")
age_dist     = get_distribution(df_transactions, "age_group")

results["transactions"] = {
    "source": "purchase_history",
    **txn_checks,
    "dtypes":               dtype_map(purchase_cols, purchase_schema),
    "payment_distribution": payment_dist,
    "age_distribution":     age_dist,
}

print(f"  rows={results['transactions']['total_rows']:,}  "
      f"range_checks={len(results['transactions']['range_checks'])}")
print(f"  Cleaned: {txn_checks['cleaning_summary']['final_rows']:,} rows retained "
      f"({txn_checks['cleaning_summary']['retention_rate_pct']}%)")

# Write cleaned data to silver table
df_transactions_clean.write.mode("overwrite").saveAsTable("workspace.recomart_silver.purchase_history_cleaned")
print("  ✅ Cleaned data saved to: workspace.recomart_silver.purchase_history_cleaned")

In [0]:
print("Profiling: product_catalog ...")

df_catalog   = spark.table("workspace.recomart_bronze.product_catalog")
catalog_cols = df_catalog.columns
catalog_schema = df_catalog.schema

# Cast ArrayType tags column to string in Spark (no pandas .apply needed)
if "tags" in catalog_cols:
    df_catalog = df_catalog.withColumn("tags", F.col("tags").cast("string"))

VALID_CATS     = ["Electronics","Clothing","Books","Home & Kitchen",
                   "Sports","Beauty","Toys","Grocery"]
VALID_SUB_CATS = ["Premium","Budget","Mid-range"]
VALID_BRANDS   = ["BrandAlpha","BrandBeta","BrandGamma","BrandDelta",
                   "BrandEpsilon","BrandZeta","BrandEta","BrandTheta"]

cat_checks, df_catalog_clean = run_all_checks(
    df           = df_catalog,
    cols         = catalog_cols,
    numeric_cols = ["price", "stock", "rating", "review_count"],
    cardinality_cols = ["category", "sub_category", "brand"],
    domain_checks = [
        {
            "name":      "category values",
            "condition": ~F.col("category").isin(VALID_CATS),
        },
        {
            "name":      "sub_category values",
            "condition": ~F.col("sub_category").isin(VALID_SUB_CATS),
        },
        {
            "name":      "brand values",
            "condition": ~F.col("brand").isin(VALID_BRANDS),
        },
    ],
    range_checks_def = [
        {"col": "price",        "lo": 0.01, "hi": 100000},
        {"col": "stock",        "lo": 0,    "hi": 100000},
        {"col": "rating",       "lo": 1.0,  "hi": 5.0},
        {"col": "review_count", "lo": 0,    "hi": 1000000},
    ],
    pk_col = "item_id",
)

# Filter to keep only active products
rows_before_active_filter = df_catalog_clean.count()
df_catalog_clean = df_catalog_clean.filter(F.col("is_active") == True)
rows_after_active_filter = df_catalog_clean.count()
inactive_removed = rows_before_active_filter - rows_after_active_filter

# Update cleaning summary to reflect inactive products removal
cat_checks["cleaning_summary"]["final_rows"] = rows_after_active_filter
cat_checks["cleaning_summary"]["total_removed"] += inactive_removed
cat_checks["cleaning_summary"]["removed_by_reason"]["inactive"] = inactive_removed
cat_checks["cleaning_summary"]["retention_rate_pct"] = pct(rows_after_active_filter, cat_checks["cleaning_summary"]["original_rows"])

cat_dist = get_distribution(df_catalog, "category")

# Count active items — single filter+count, done separately (lightweight)
active_items = df_catalog_clean.count()

results["catalog"] = {
    "source": "product_catalog",
    **cat_checks,
    "dtypes":                dtype_map(catalog_cols, catalog_schema),
    "category_distribution": cat_dist,
    "active_items":          active_items,
    "inactive_removed":      inactive_removed,
}

print(f"  rows={results['catalog']['total_rows']:,}  "
      f"range_checks={len(results['catalog']['range_checks'])}")
print(f"  Cleaned: {cat_checks['cleaning_summary']['final_rows']:,} rows retained "
      f"({cat_checks['cleaning_summary']['retention_rate_pct']}%)")
print(f"  Inactive products removed: {inactive_removed}")
print(f"  Active products: {active_items}")

# Write cleaned data to silver table
df_catalog_clean.write.mode("overwrite").saveAsTable("workspace.recomart_silver.product_catalog_cleaned")
print("  ✅ Cleaned data saved to: workspace.recomart_silver.product_catalog_cleaned")

In [0]:
print("Profiling: external_signals ...")

df_external   = spark.table("workspace.recomart_bronze.external_signals")
external_cols = df_external.columns
external_schema = df_external.schema

ext_checks, df_external_clean = run_all_checks(
    df           = df_external,
    cols         = external_cols,
    numeric_cols = ["popularity_score", "avg_sentiment",
                     "trend_score", "search_volume_7d", "return_rate"],
    cardinality_cols = ["item_id"],
    domain_checks = [],
    range_checks_def = [
        {"col": "popularity_score", "lo": 0.0,  "hi": 1.0},
        {"col": "avg_sentiment",    "lo": -1.0,  "hi": 1.0},
        {"col": "trend_score",      "lo": 0.0,  "hi": 100.0},
        {"col": "return_rate",      "lo": 0.0,  "hi": 1.0},
    ],
    pk_col = "item_id",
)

results["external"] = {
    "source": "external_signals",
    **ext_checks,
    "dtypes": dtype_map(external_cols, external_schema),
}

print(f"  rows={results['external']['total_rows']:,}  "
      f"range_checks={len(results['external']['range_checks'])}")
print(f"  Cleaned: {ext_checks['cleaning_summary']['final_rows']:,} rows retained "
      f"({ext_checks['cleaning_summary']['retention_rate_pct']}%)")

# Write cleaned data to silver table
df_external_clean.write.mode("overwrite").saveAsTable("workspace.recomart_silver.external_signals_cleaned")
print("  ✅ Cleaned data saved to: workspace.recomart_silver.external_signals_cleaned")

In [0]:
print("Running cross-source referential integrity checks on cleaned data ...")

# Use the cleaned dataframes for referential integrity checks
catalog_ids  = df_catalog_clean.select("item_id").distinct()
external_ids = df_external_clean.select("item_id").distinct()
click_ids    = df_clickstream_clean.select("item_id").distinct()
txn_ids      = df_transactions_clean.select("item_id").distinct()
click_users  = df_clickstream_clean.select("user_id").distinct()
txn_users    = df_transactions_clean.select("user_id").distinct()

orphan_click  = click_ids.join(catalog_ids,  "item_id", "left_anti").count()
orphan_txn    = txn_ids.join(catalog_ids,    "item_id", "left_anti").count()
missing_sig   = catalog_ids.join(external_ids,"item_id","left_anti").count()
user_overlap  = click_users.join(txn_users,   "user_id", "inner").count()

results["cross_source"] = {
    "clickstream_items_not_in_catalog":  {"count": orphan_click, "status": "PASS" if orphan_click == 0 else "WARN"},
    "transaction_items_not_in_catalog":  {"count": orphan_txn,   "status": "PASS" if orphan_txn   == 0 else "WARN"},
    "catalog_items_missing_ext_signals": {"count": missing_sig,  "status": "PASS" if missing_sig  == 0 else "WARN"},
    "user_overlap_click_txn":            user_overlap,
}

print(f"  Referential integrity done — orphan_click={orphan_click}, "
      f"orphan_txn={orphan_txn}, missing_signals={missing_sig}")
print("  ✅ All referential integrity checks use cleaned data")

In [0]:
# ════════════════════════════════════════════════════════════════════════════
# DATA CLEANING SUMMARY TABLE
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 80)
print("DATA CLEANING SUMMARY")
print("=" * 80)

cleaning_data = []
for ds_name, ds_key in [("Clickstream", "clickstream"), 
                         ("Transactions", "transactions"), 
                         ("Catalog", "catalog"), 
                         ("External", "external")]:
    summary = results[ds_key]["cleaning_summary"]
    cleaning_data.append({
        "Data Source": ds_name,
        "Original Rows": summary["original_rows"],
        "Final Rows": summary["final_rows"],
        "Total Removed": summary["total_removed"],
        "Missing": summary["removed_by_reason"]["missing"],
        "Domain": summary["removed_by_reason"]["domain"],
        "Range": summary["removed_by_reason"]["range"],
        "Duplicates": summary["removed_by_reason"]["duplicates"],
        "Inactive": summary["removed_by_reason"].get("inactive", 0),
        "Retention %": summary["retention_rate_pct"]
    })

df_cleaning_summary = spark.createDataFrame(cleaning_data)
display(df_cleaning_summary)

# Calculate totals
total_original = sum(d["Original Rows"] for d in cleaning_data)
total_final = sum(d["Final Rows"] for d in cleaning_data)
total_removed = sum(d["Total Removed"] for d in cleaning_data)
total_inactive = sum(d["Inactive"] for d in cleaning_data)

print(f"\n📊 OVERALL STATISTICS:")
print(f"  Total Original Rows: {total_original:,}")
print(f"  Total Final Rows:    {total_final:,}")
print(f"  Total Removed:       {total_removed:,}")
print(f"    - Inactive:        {total_inactive}")
print(f"  Overall Retention:   {pct(total_final, total_original)}%")
print("\n✅ All cleaned tables saved to workspace.recomart_silver catalog")

In [0]:
# Additional imports for feature preparation and EDA
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import warnings
import os

warnings.filterwarnings("ignore")

# Plot output directory
PLOT_DIR = "/Workspace/Users/2025ae05764@wilp.bits-pilani.ac.in/RecomartEcommerce/2_medallion_processing/plots"
os.makedirs(PLOT_DIR, exist_ok=True)

print("✅ Additional imports loaded")
print(f"📁 Plot directory: {PLOT_DIR}")

In [0]:
# Shared plot style
BLUE   = "#2F7FBF"
NAVY   = "#1B3A6B"
ORANGE = "#E87040"
GREEN  = "#2E8B57"
GRAY   = "#AAAAAA"

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F8FAFD",
    "axes.edgecolor":   "#CCCCCC",
    "axes.grid":        True,
    "grid.color":       "#E0E6F0",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.titleweight": "bold",
    "axes.titlepad":    10,
    "axes.labelsize":   9,
    "xtick.labelsize":  8,
    "ytick.labelsize":  8,
})

def save_plot(name):
    path = f"{PLOT_DIR}/{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  📊 {name}.png saved")
    return path

print("✅ Plot configuration applied")

In [0]:
# Convert cleaned Spark DataFrames to Pandas for feature engineering
print("Converting Spark DataFrames to Pandas...")

df_click = df_clickstream_clean.toPandas()
df_txn   = df_transactions_clean.toPandas()
df_cat   = df_catalog_clean.toPandas()
df_ext   = df_external_clean.toPandas()

print(f"  Clickstream : {len(df_click):,} rows")
print(f"  Transactions: {len(df_txn):,} rows")
print(f"  Catalog     : {len(df_cat):,} rows")
print(f"  External    : {len(df_ext):,} rows")
print("\n✅ Data converted to Pandas")

In [0]:
print("\nEncoding categorical columns ...")

# --- Clickstream ---
le_event  = LabelEncoder()
le_device = LabelEncoder()
df_click["event_type_enc"] = le_event.fit_transform(df_click["event_type"])
df_click["device_enc"]     = le_device.fit_transform(df_click["device"])

# One-hot for event_type (used in downstream feature tables)
ohe_events = pd.get_dummies(df_click["event_type"], prefix="evt")
df_click = pd.concat([df_click, ohe_events], axis=1)

# --- Transactions ---
le_pay    = LabelEncoder()
le_gender = LabelEncoder()
le_age    = LabelEncoder()
df_txn["payment_enc"] = le_pay.fit_transform(df_txn["payment_method"])
df_txn["gender_enc"]  = le_gender.fit_transform(df_txn["gender"])
df_txn["age_enc"]     = le_age.fit_transform(df_txn["age_group"])

# --- Catalog ---
le_cat   = LabelEncoder()
le_brand = LabelEncoder()
le_sub   = LabelEncoder()
df_cat["category_enc"]     = le_cat.fit_transform(df_cat["category"])
df_cat["brand_enc"]        = le_brand.fit_transform(df_cat["brand"])
df_cat["sub_category_enc"] = le_sub.fit_transform(df_cat["sub_category"])

print("✅ Label encoding done for event_type, device, payment, gender, age, category, brand")
print(f"  Clickstream encoded columns: event_type_enc, device_enc, {list(ohe_events.columns)}")
print(f"  Transactions encoded columns: payment_enc, gender_enc, age_enc")
print(f"  Catalog encoded columns: category_enc, brand_enc, sub_category_enc")

In [0]:
print("\nNormalizing numerical columns ...")
scaler = MinMaxScaler()

# Transactions — price columns
for col in ["unit_price", "total_price"]:
    df_txn[f"{col}_norm"] = scaler.fit_transform(df_txn[[col]])
    df_txn[f"{col}_log"]  = np.log1p(df_txn[col])

df_txn["quantity_norm"] = scaler.fit_transform(df_txn[["quantity"]])

# Catalog — price, stock, review_count
for col in ["price", "stock", "review_count"]:
    df_cat[f"{col}_norm"] = scaler.fit_transform(df_cat[[col]])
    df_cat[f"{col}_log"]  = np.log1p(df_cat[col])

print("✅ Min-Max scaling + log1p transform applied")
print(f"  Transactions normalized: unit_price, total_price, quantity")
print(f"  Catalog normalized: price, stock, review_count")
print(f"  Added columns: *_norm (scaled [0,1]) and *_log (log1p transformed)")

In [0]:
print("\nBuilding user-item interaction matrix ...")

# Assign weights to events: purchase=5, add_to_cart=3, click=2, view=1
EVENT_WEIGHTS = {"purchase": 5, "add_to_cart": 3, "click": 2, "view": 1}
df_click["weight"] = df_click["event_type"].map(EVENT_WEIGHTS)

# Aggregate: sum of weights per (user, item)
ui_agg = (df_click.groupby(["user_id", "item_id"])["weight"]
                   .sum().reset_index()
                   .rename(columns={"weight": "interaction_score"}))

# Pivot to matrix (subset top users & items for manageability)
top_users = ui_agg.groupby("user_id")["interaction_score"].sum().nlargest(50).index
top_items = ui_agg.groupby("item_id")["interaction_score"].sum().nlargest(100).index
ui_sub    = ui_agg[ui_agg["user_id"].isin(top_users) & ui_agg["item_id"].isin(top_items)]
ui_matrix = ui_sub.pivot(index="user_id", columns="item_id", values="interaction_score").fillna(0)

n_users, n_items = ui_matrix.shape
sparsity = 1 - (ui_matrix > 0).values.sum() / (n_users * n_items)

print(f"  📈 Matrix shape  : {n_users} users × {n_items} items")
print(f"  🕳️  Sparsity      : {sparsity:.1%}")
print(f"  ✨ Total non-zero interactions: {int((ui_matrix > 0).values.sum()):,}")
print(f"\n📊 Event weights used:")
for event, weight in EVENT_WEIGHTS.items():
    print(f"     {event:15s} = {weight}")

# Display sample
print(f"\n🔍 Sample of interaction matrix (first 5x5):")
display(ui_matrix.iloc[:5, :5])

In [0]:
print("\n📊 Generating EDA plots ...\n")
print("Plot 1: Event-Type Distribution")

fig, ax = plt.subplots(figsize=(7, 4))
evt_counts = df_click["event_type"].value_counts()
bars = ax.bar(evt_counts.index, evt_counts.values,
              color=[NAVY, BLUE, ORANGE, GREEN], edgecolor="white", width=0.55)
for bar, val in zip(bars, evt_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f"{val:,}", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax.set_title("Event-Type Distribution — Clickstream")
ax.set_xlabel("Event Type")
ax.set_ylabel("Number of Events")
ax.set_ylim(0, evt_counts.max() * 1.15)
save_plot("01_event_type_distribution")

In [0]:
print("Plot 2: Item Popularity Distribution")

fig, ax = plt.subplots(figsize=(7, 4))
item_pop = df_click.groupby("item_id")["weight"].sum().sort_values(ascending=False)
ax.hist(item_pop.values, bins=40, color=BLUE, edgecolor="white", alpha=0.88)
ax.axvline(item_pop.median(), color=ORANGE, linewidth=1.8, linestyle="--",
           label=f"Median = {item_pop.median():.0f}")
ax.axvline(item_pop.mean(),   color=NAVY,   linewidth=1.8, linestyle="-.",
           label=f"Mean = {item_pop.mean():.0f}")
ax.set_title("Item Popularity Distribution (Weighted Interaction Score)")
ax.set_xlabel("Total Interaction Score per Item")
ax.set_ylabel("Number of Items")
ax.legend(fontsize=8)
save_plot("02_item_popularity_distribution")

In [0]:
print("Plot 3: Purchase Price Distribution (Raw + Log)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(df_txn["unit_price"], bins=40, color=ORANGE, edgecolor="white", alpha=0.85)
ax1.set_title("Unit Price — Raw")
ax1.set_xlabel("Price (₹)")
ax1.set_ylabel("Frequency")

ax2.hist(df_txn["unit_price_log"], bins=40, color=GREEN, edgecolor="white", alpha=0.85)
ax2.set_title("Unit Price — Log Transformed")
ax2.set_xlabel("log1p(Price)")
ax2.set_ylabel("Frequency")
fig.suptitle("Purchase Price Distribution", fontweight="bold", fontsize=12, y=1.01)
save_plot("03_price_distribution")

In [0]:
print("Plot 4: User Activity Distribution")

fig, ax = plt.subplots(figsize=(7, 4))
user_activity = df_click.groupby("user_id").size()
ax.hist(user_activity.values, bins=35, color=NAVY, edgecolor="white", alpha=0.85)
ax.axvline(user_activity.median(), color=ORANGE, linewidth=1.8, linestyle="--",
           label=f"Median = {user_activity.median():.0f}")
ax.set_title("User Activity Distribution (Events per User)")
ax.set_xlabel("Number of Events")
ax.set_ylabel("Number of Users")
ax.legend(fontsize=8)
save_plot("04_user_activity_distribution")

In [0]:
print("Plot 5: Product Rating Distribution")

fig, ax = plt.subplots(figsize=(7, 4))
rating_bins = [1,1.5,2,2.5,3,3.5,4,4.5,5,5.01]
rating_labels = ["1.0","1.5","2.0","2.5","3.0","3.5","4.0","4.5","5.0"]
df_cat["rating_bin"] = pd.cut(df_cat["rating"], bins=rating_bins,
                               labels=rating_labels, include_lowest=True)
rc = df_cat["rating_bin"].value_counts().sort_index()
bars = ax.bar(rc.index.astype(str), rc.values, color=BLUE, edgecolor="white", width=0.65)
for b, v in zip(bars, rc.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, str(v),
            ha="center", va="bottom", fontsize=8)
ax.set_title("Product Rating Distribution — Catalog")
ax.set_xlabel("Rating Range")
ax.set_ylabel("Number of Products")
save_plot("05_product_rating_distribution")

In [0]:
print("Plot 6: User-Item Matrix Sparsity Heatmap")

fig, ax = plt.subplots(figsize=(10, 5))
mat_display = (ui_matrix > 0).astype(int)
im = ax.imshow(mat_display.values, aspect="auto", cmap="Blues",
               interpolation="nearest", vmin=0, vmax=1)
ax.set_title(f"User-Item Interaction Matrix — Sparsity: {sparsity:.1%}\n"
             f"({n_users} users × {n_items} items shown)")
ax.set_xlabel("Item Index")
ax.set_ylabel("User Index")
ax.set_xticks([])
ax.set_yticks([])
cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.set_ticks([0, 1])
cb.set_ticklabels(["No interaction", "Interacted"])
cb.ax.tick_params(labelsize=8)
save_plot("06_sparsity_heatmap")

In [0]:
print("Plot 7: Top-20 Most Purchased Items")

fig, ax = plt.subplots(figsize=(8, 6))
top20 = (df_txn.groupby("item_id")["quantity"].sum()
               .nlargest(20).sort_values())
colors_bar = [NAVY if i < 5 else (BLUE if i < 15 else ORANGE)
              for i in range(len(top20))]
ax.barh(top20.index, top20.values, color=colors_bar, edgecolor="white", height=0.7)
for i, (idx, val) in enumerate(top20.items()):
    ax.text(val + 0.3, i, str(val), va="center", fontsize=8)
ax.set_title("Top-20 Most Purchased Items (Total Quantity Sold)")
ax.set_xlabel("Total Units Sold")
ax.set_ylabel("Item ID")
ax.set_xlim(0, top20.max() * 1.12)
save_plot("07_top20_purchased_items")

In [0]:
print("Plot 8: Age Group vs Event Type Heatmap")

fig, ax = plt.subplots(figsize=(8, 4.5))
# Merge clickstream with transactions to get age_group per user
user_age = df_txn[["user_id","age_group"]].drop_duplicates("user_id")
df_merged = df_click.merge(user_age, on="user_id", how="left")
df_merged["age_group"].fillna("Unknown", inplace=True)

pivot = (df_merged.groupby(["age_group","event_type"])
                  .size().unstack(fill_value=0))
# Normalize rows to show proportions
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

im2 = ax.imshow(pivot_norm.values, cmap="YlOrRd", aspect="auto", vmin=0)
ax.set_xticks(range(len(pivot_norm.columns)))
ax.set_yticks(range(len(pivot_norm.index)))
ax.set_xticklabels(pivot_norm.columns, rotation=25, ha="right")
ax.set_yticklabels(pivot_norm.index)
for i in range(len(pivot_norm.index)):
    for j in range(len(pivot_norm.columns)):
        val = pivot_norm.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=8, color="black" if val < 0.4 else "white")
cb2 = plt.colorbar(im2, ax=ax, fraction=0.025, pad=0.02)
cb2.set_label("Proportion of Events", fontsize=8)
ax.set_title("Age Group vs Event Type — Normalized Interaction Heatmap")
ax.set_xlabel("Event Type")
ax.set_ylabel("Age Group")
save_plot("08_age_group_event_heatmap")

print("\n✅ All 8 EDA plots generated successfully!")

# Feature Preparation & EDA Complete! 🎯

## What Was Accomplished

### 1. Categorical Encoding
* **Label Encoding**: Converted categorical variables to numeric codes
  * Clickstream: event_type_enc, device_enc
  * Transactions: payment_enc, gender_enc, age_enc
  * Catalog: category_enc, brand_enc, sub_category_enc
* **One-Hot Encoding**: Created binary columns for event_type (evt_*)

### 2. Numerical Normalization
* **Min-Max Scaling** ([0, 1] range):
  * Transactions: unit_price_norm, total_price_norm, quantity_norm
  * Catalog: price_norm, stock_norm, review_count_norm
* **Log Transformation** (log1p for skewed distributions):
  * Transactions: unit_price_log, total_price_log
  * Catalog: price_log, stock_log, review_count_log

### 3. User-Item Interaction Matrix
* **Event Weights**: purchase=5, add_to_cart=3, click=2, view=1
* **Matrix Size**: 50 users × 100 items (top users/items)
* **Sparsity**: Calculated to measure data density
* **Purpose**: Foundation for collaborative filtering recommendation models

### 4. Exploratory Data Analysis (8 Plots)
1. ✅ **Event-Type Distribution** - Shows balance of user interactions
2. ✅ **Item Popularity** - Distribution of interaction scores across items
3. ✅ **Purchase Price** - Raw and log-transformed price distributions
4. ✅ **User Activity** - Events per user distribution
5. ✅ **Product Ratings** - Rating distribution in catalog
6. ✅ **Sparsity Heatmap** - Visualization of user-item interactions
7. ✅ **Top-20 Items** - Most purchased products by quantity
8. ✅ **Age vs Event Type** - Heatmap showing interaction patterns by demographics

### 5. Output Artifacts
* **Plots**: 8 PNG files saved to `/plots/` directory
* **Prepared DataFrames**: With encoded and normalized features ready for ML modeling
* **Interaction Matrix**: User-item matrix for recommendation algorithms

---

## Next Steps for ML Model Development

1. **Feature Engineering**: Create additional features from prepared data
2. **Train-Test Split**: Temporal or random splitting for model validation
3. **Model Training**: 
   * Collaborative Filtering (Matrix Factorization, ALS)
   * Content-Based Filtering (using catalog features)
   * Hybrid Models (combining both approaches)
4. **Model Evaluation**: Precision@K, Recall@K, NDCG, MAP
5. **Hyperparameter Tuning**: Optimize model performance
6. **Production Deployment**: Serve recommendations via API

**All data is now ready for machine learning model development!** 🚀

In [0]:
# ════════════════════════════════════════════════════════════════════════════
# OVERALL SCORECARD
# ════════════════════════════════════════════════════════════════════════════
def count_checks(res: Dict) -> tuple:
    passed = failed = warned = 0
    for key in ["domain_checks", "range_checks"]:
        for c in res.get(key, []):
            if   c["status"] == "PASS": passed += 1
            elif c["status"] == "FAIL": failed += 1
            else:                       warned += 1
    dup = res.get("key_dup", {})
    if   dup.get("status") == "PASS": passed += 1
    elif dup.get("status") == "WARN": warned += 1
    return passed, failed, warned

scorecard = {}
for ds in ["clickstream", "transactions", "catalog", "external"]:
    p, f, w = count_checks(results[ds])
    total_missing = sum(r["missing_count"] for r in results[ds]["missing"])
    scorecard[ds] = {
        "checks_passed":       p,
        "checks_failed":       f,
        "checks_warned":       w,
        "total_missing_values":total_missing,
        "overall":             "PASS" if f == 0 else "FAIL",
    }

results["scorecard"]    = scorecard
results["generated_at"] = datetime.now().isoformat()

# ════════════════════════════════════════════════════════════════════════════
# SAVE TO DELTA TABLE
# ════════════════════════════════════════════════════════════════════════════
results_json_str = json.dumps(results, default=str)

df_out = spark.createDataFrame(
    [{"validation_json": results_json_str,
      "generated_at":    results["generated_at"]}]
)

df_out.write \
      .mode("overwrite") \
      .saveAsTable("workspace.recomart_silver.validation_data_json")

display(spark.table("workspace.recomart_silver.validation_data_json"))

# ── Console scorecard ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("RECOMART DATA QUALITY VALIDATION SUMMARY")
print("=" * 65)
for ds, sc in scorecard.items():
    print(f"\n[{ds.upper()}]  Overall: {sc['overall']}")
    print(f"  Checks passed : {sc['checks_passed']}")
    print(f"  Checks failed : {sc['checks_failed']}")
    print(f"  Checks warned : {sc['checks_warned']}")
    print(f"  Missing values: {sc['total_missing_values']}")
print("\nDone.")

# Validation & Cleaning Complete! ✅

## What Was Accomplished

This notebook successfully validated and cleaned 4 RecoMart data sources:

### 1. Data Validation Results
* **Clickstream**: 5,000 rows → All checks PASSED
* **Transactions**: 1,500 rows → All checks PASSED (1 warning)
* **Catalog**: 500 rows → All checks PASSED
* **External Signals**: 500 rows → All checks PASSED

### 2. Data Cleaning Results
* **Total Original Rows**: 7,500
* **Total Final Rows**: 7,146
* **Total Removed**: 354 rows
* **Overall Retention**: 95.28%

**Breakdown by Data Source**:
| Data Source | Original | Final | Removed | Retention |
|------------|----------|-------|---------|----------|
| Clickstream | 5,000 | 4,750 | 250 | 95.0% |
| Transactions | 1,500 | 1,396 | 104 | 93.07% |
| Catalog | 500 | 500 | 0 | 100% |
| External | 500 | 500 | 0 | 100% |

**Rows Removed By Reason**:
* Missing values: 311 rows (250 clickstream + 61 transactions)
* Duplicates: 43 rows (transactions only)
* Domain violations: 0 rows
* Range violations: 0 rows

### 3. Output Tables Created

Cleaned data saved to **workspace.recomart_silver** catalog:
1. ✅ `user_interaction_logs_cleaned`
2. ✅ `purchase_history_cleaned`
3. ✅ `product_catalog_cleaned`
4. ✅ `external_signals_cleaned`
5. ✅ `validation_data_json` (validation results)

### 4. Referential Integrity
* ✅ All clickstream items exist in catalog
* ✅ All transaction items exist in catalog
* ✅ All catalog items have external signals
* ✅ 200 users overlap between clickstream and transactions

---

## Next Steps

Use the cleaned tables for downstream analytics and machine learning:
```sql
SELECT * FROM workspace.recomart_silver.user_interaction_logs_cleaned;
SELECT * FROM workspace.recomart_silver.purchase_history_cleaned;
SELECT * FROM workspace.recomart_silver.product_catalog_cleaned;
SELECT * FROM workspace.recomart_silver.external_signals_cleaned;
```

In [0]:
# Export validation results to JSON file
import json

# Define output path
output_path = "/Workspace/Users/2025ae05764@wilp.bits-pilani.ac.in/RecomartEcommerce/2_medallion_processing/validation_results.json"

# Write results to JSON file
with open(output_path.replace('/Workspace', '/Workspace'), 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"✅ Validation results exported to: {output_path}")
print(f"\nFile size: {len(json.dumps(results, default=str))} bytes")
print(f"\nSummary:")
print(f"  - Clickstream: {results['clickstream']['total_rows']:,} rows")
print(f"  - Transactions: {results['transactions']['total_rows']:,} rows")
print(f"  - Catalog: {results['catalog']['total_rows']:,} rows")
print(f"  - External: {results['external']['total_rows']:,} rows")
print(f"\nOverall Status:")
for ds, sc in results['scorecard'].items():
    print(f"  - {ds.upper()}: {sc['overall']}")

In [0]:
# Display file content and download instructions
import json

file_path = "/Workspace/Users/2025ae05764@wilp.bits-pilani.ac.in/RecomartEcommerce/2_medallion_processing/validation_results.json"

# Read and display the JSON content
with open(file_path, 'r') as f:
    content = f.read()
    data = json.loads(content)

print("📥 DOWNLOAD OPTIONS:")
print("=" * 70)
print("\n1. WORKSPACE FILE BROWSER:")
print("   • Click 'Workspace' in the left sidebar")
print("   • Navigate to: RecomartEcommerce/2_medallion_processing/")
print("   • Right-click 'validation_results.json' → Download")
print("\n2. VIEW CONTENT BELOW:")
print("   • Copy the JSON output from the cell result below")
print("\n" + "=" * 70)
print("\n📊 FILE PREVIEW (first 500 characters):")
print(content[:500] + "...\n")
print(f"\n📏 Total file size: {len(content):,} bytes")
print(f"📦 Total data sources validated: {len(data.get('scorecard', {}))}")

# Display full JSON in a collapsible format
print("\n" + "=" * 70)
print("FULL JSON CONTENT:")
print("=" * 70)
print(json.dumps(data, indent=2, default=str))

In [0]:
print("Verify versioning in githb")